In [1]:
%pip install pandas openpyxl matplotlib seaborn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import numpy as np

sns.set_theme(style="whitegrid", palette="Set2")

# ── Output folder for vector figures ──────────────────────────────────────
FIG_DIR = Path("figures")
FIG_DIR.mkdir(exist_ok=True)

def save_fig(name: str):
    """Save current figure as SVG and PDF into FIG_DIR."""
    plt.savefig(FIG_DIR / f"{name}.svg", bbox_inches="tight")
    plt.savefig(FIG_DIR / f"{name}.pdf", bbox_inches="tight")
    print(f"  Saved: figures/{name}.svg + .pdf")

In [3]:
# Path to your Excel file
file_path = "test_data.xlsx"  # change this to your actual file name

# If your data is in the first sheet
df = pd.read_excel(file_path)

# If you need a specific sheet:
# df = pd.read_excel(file_path, sheet_name="Sheet1")

# Take a quick look
df.head()

,Time,Condition,Value
0,0,Control,0.10
1,0,Treated,0.12
2,1,Control,0.20
3,1,Treated,0.35
4,2,Control,0.32


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(df["Time"], df["Value"], marker="o")
plt.xlabel("Time")
plt.ylabel("Value")
plt.title("Value over Time")
plt.tight_layout()
save_fig("01_line_all")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
for condition, sub_df in df.groupby("Condition"):
    plt.plot(sub_df["Time"], sub_df["Value"], marker="o", label=condition)
plt.xlabel("Time")
plt.ylabel("Value")
plt.title("Value over Time by Condition")
plt.legend()
plt.tight_layout()
save_fig("02_line_by_condition")
plt.show()

In [6]:
# Compute means (and optionally std)
summary = df.groupby("Condition")["Value"].agg(["mean", "std"]).reset_index()
summary

,Condition,mean,std
0,Control,0.255,0.132035
1,Treated,0.405,0.226053


In [ ]:
summary = df.groupby("Condition")["Value"].agg(["mean", "std"]).reset_index()

plt.figure(figsize=(6, 4))
plt.bar(summary["Condition"], summary["mean"], yerr=summary["std"], capsize=5)
plt.xlabel("Condition")
plt.ylabel("Mean Value")
plt.title("Mean Value ± SD by Condition")
plt.tight_layout()
save_fig("03_bar_mean_sd")
plt.show()

In [ ]:
plt.figure(figsize=(6, 5))
sns.scatterplot(data=df, x="Time", y="Value", hue="Condition", s=60)
plt.title("Value vs Time (scatter)")
plt.tight_layout()
save_fig("04_scatter")
plt.show()

---
## Additional analysis graphs

In [ ]:
# Error band: mean ± std as shaded area over time
stats = df.groupby(["Time", "Condition"])["Value"].agg(["mean", "std"]).reset_index()

fig, ax = plt.subplots(figsize=(8, 5))
for cond, grp in stats.groupby("Condition"):
    ax.plot(grp["Time"], grp["mean"], marker="o", label=cond)
    ax.fill_between(grp["Time"],
                    grp["mean"] - grp["std"],
                    grp["mean"] + grp["std"],
                    alpha=0.25)
ax.set_xlabel("Time")
ax.set_ylabel("Value")
ax.set_title("Mean ± SD over Time by Condition")
ax.legend()
plt.tight_layout()
save_fig("05_error_band")
plt.show()

In [ ]:
# Box plot: value distribution per condition
fig, ax = plt.subplots(figsize=(6, 5))
sns.boxplot(data=df, x="Condition", y="Value",
            width=0.5, flierprops=dict(marker="o", markersize=4), ax=ax)
ax.set_title("Value Distribution by Condition (box plot)")
ax.set_xlabel("Condition")
ax.set_ylabel("Value")
plt.tight_layout()
save_fig("06_boxplot")
plt.show()

In [ ]:
# Violin plot with individual data points overlaid
fig, ax = plt.subplots(figsize=(6, 5))
sns.violinplot(data=df, x="Condition", y="Value", inner="box", ax=ax)
sns.stripplot(data=df, x="Condition", y="Value",
              color="black", size=4, alpha=0.5, jitter=True, ax=ax)
ax.set_title("Value Distribution by Condition (violin)")
ax.set_xlabel("Condition")
ax.set_ylabel("Value")
plt.tight_layout()
save_fig("07_violin")
plt.show()

In [ ]:
# KDE density curves per condition
fig, ax = plt.subplots(figsize=(7, 4))
for cond, grp in df.groupby("Condition"):
    sns.kdeplot(grp["Value"], label=cond, fill=True, alpha=0.3, ax=ax)
ax.set_title("Value Density Distribution by Condition")
ax.set_xlabel("Value")
ax.set_ylabel("Density")
ax.legend()
plt.tight_layout()
save_fig("08_kde_density")
plt.show()

In [ ]:
# Heatmap: mean value at each Time × Condition
pivot = df.groupby(["Time", "Condition"])["Value"].mean().unstack()

fig, ax = plt.subplots(figsize=(6, 4))
sns.heatmap(pivot, annot=True, fmt=".2f", cmap="YlOrRd",
            linewidths=0.5, ax=ax)
ax.set_title("Mean Value Heatmap (Time × Condition)")
ax.set_xlabel("Condition")
ax.set_ylabel("Time")
plt.tight_layout()
save_fig("09_heatmap")
plt.show()

In [ ]:
# Regression: trend line with confidence interval per condition
fig, ax = plt.subplots(figsize=(7, 5))
for cond, grp in df.groupby("Condition"):
    sns.regplot(data=grp, x="Time", y="Value",
                scatter_kws={"s": 40, "alpha": 0.7},
                line_kws={"linewidth": 2},
                label=cond, ax=ax)
ax.set_title("Value vs Time with Regression Line")
ax.set_xlabel("Time")
ax.set_ylabel("Value")
ax.legend()
plt.tight_layout()
save_fig("10_regression")
plt.show()